# AISO Fraud Benchmark — YelpChi & Amazon

두 개의 독립된 연구 질문을 각 데이터셋에서 검증한다.

| 섹션 | 연구 질문 | 변인 | 고정 |
|------|----------|------|------|
| **A (Exp 1b 재현)** | AISO가 그래프 샘플러로서 가치있는가? | 노드 선택 전략 | 피처 전체 고정 |
| **B (Exp 7 재현)** | AISO가 피처 엔지니어링을 자동화하는가? | 피처 선택 전략 | 스코어 기반 top-N 고정 |

**Section A**: Random | Greedy(top-N) | Cluster-uniform | AISO-node(Rand M) | AISO-node(Smart M)  
**Section B**: MI top-K | mRMR top-K | AISO(Rand M) | AISO(Smart M)

In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.sparse import issparse
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import copy, time
import matplotlib.pyplot as plt
from pathlib import Path

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE         = Path(r'C:\Users\kevin\OneDrive\Desktop\AISO')
GAMMA        = 0.5
GNN_SEED     = 42
SEEDS        = [0, 7, 42, 77, 123]
GNN_BUDGET   = 3    # GNN calls per seed (reduced for speed)
DIV_INTERVAL = 15
TEST_RATIO   = 0.2
GNN_EPOCHS   = 150  # reduced from 200
GNN_PATIENCE = 15

print(f'Device: {DEVICE}')

c:\Users\kevin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


---
## 1. 데이터 로드

In [ ]:
from scipy.sparse import csr_matrix, diags as sp_diags

DATASET_CONFIGS = {
    'YelpChi': dict(
        mat_path   = BASE / 'YelpChi' / 'YelpChi.mat',
        feat_key   = 'features',
        label_key  = 'label',
        adj_key    = 'homo',
        K_SELECT   = 8,    # Exp 7: 8 of 32 raw features
        N_TYPES    = 6,    # Exp 7: feature clusters
        K_DOM      = 5,    # Exp 1b: 5 of ~10 domain features
        N_DOM_TYPES= 4,    # Exp 1b: domain feature clusters
        N_ILLICIT  = 500,
        N_LICIT    = 5000,
    ),
    'Amazon': dict(
        mat_path   = BASE / 'Amazon' / 'Amazon.mat',
        feat_key   = 'features',
        label_key  = 'label',
        adj_key    = 'homo',
        K_SELECT   = 6,    # Exp 7: 6 of 25 raw features
        N_TYPES    = 5,    # Exp 7: feature clusters
        K_DOM      = 5,    # Exp 1b: 5 of ~10 domain features
        N_DOM_TYPES= 4,    # Exp 1b: domain feature clusters
        N_ILLICIT  = 200,
        N_LICIT    = 2000,
    ),
}

def load_dataset(cfg, seed=GNN_SEED):
    mat = sio.loadmat(str(cfg['mat_path']))

    X = mat[cfg['feat_key']]
    if issparse(X): X = X.toarray()
    X = X.astype(float)

    y = mat[cfg['label_key']].squeeze().astype(int)

    adj = mat[cfg['adj_key']]
    if issparse(adj):
        coo = adj.tocoo()
    else:
        coo = csr_matrix(adj).tocoo()
    src = coo.row.astype(np.int64)
    dst = coo.col.astype(np.int64)
    src_bi = np.concatenate([src, dst])
    dst_bi = np.concatenate([dst, src])
    mask   = src_bi != dst_bi
    src_bi, dst_bi = src_bi[mask], dst_bi[mask]
    edge_index = torch.tensor(np.stack([src_bi, dst_bi]), dtype=torch.long)
    ei_np = edge_index.numpy()

    N   = len(y)
    idx = np.arange(N)
    tr_idx, te_idx = train_test_split(idx, test_size=TEST_RATIO, stratify=y, random_state=seed)
    train_mask = np.zeros(N, dtype=bool); test_mask = np.zeros(N, dtype=bool)
    train_mask[tr_idx] = True; test_mask[te_idx] = True

    X_scaled = StandardScaler().fit_transform(X)

    train_illicit_idx = np.where(train_mask & (y == 1))[0]
    train_licit_idx   = np.where(train_mask & (y == 0))[0]

    n_licit   = min(cfg['N_LICIT'],   len(train_licit_idx))
    n_illicit = min(cfg['N_ILLICIT'], len(train_illicit_idx))
    sel_n     = np.random.RandomState(GNN_SEED).choice(train_licit_idx, n_licit, replace=False)

    X_illicit_pool = X_scaled[train_illicit_idx]
    X_tr_all       = X_scaled[train_mask]
    y_tr_all       = y[train_mask]

    illicit_ratio = (y == 1).sum() / N
    print(f'  {cfg["mat_path"].parent.name}: N={N:,}  edges(1dir)={len(src_bi)//2:,}')
    print(f'  illicit={illicit_ratio:.1%}  train_illicit={len(train_illicit_idx):,}  D={X.shape[1]}')
    print(f'  N_ILLICIT={n_illicit}  N_LICIT={n_licit}  K_SELECT={cfg["K_SELECT"]}')

    return dict(
        X_raw=X, X_scaled=X_scaled, y=y,
        edge_index=edge_index, ei_np=ei_np,
        train_mask=train_mask, test_mask=test_mask,
        train_illicit_idx=train_illicit_idx, train_licit_idx=train_licit_idx,
        sel_n=sel_n,
        X_illicit_pool=X_illicit_pool,
        X_tr_all=X_tr_all, y_tr_all=y_tr_all,
        N_NODES=N, D=X.shape[1],
        N_ILLICIT=n_illicit, N_LICIT=n_licit,
    )

print('=== YelpChi ===')
yelp_data = load_dataset(DATASET_CONFIGS['YelpChi'])
print()
print('=== Amazon ===')
amzn_data = load_dataset(DATASET_CONFIGS['Amazon'])

---
## 2. GCN + evaluate_gnn Factory

In [3]:
class GCN(torch.nn.Module):
    def __init__(self, in_ch, hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin   = torch.nn.Linear(hidden, 2)
        self.drop  = dropout

    def forward(self, x, ei):
        x = F.relu(self.conv1(x, ei))
        x = F.dropout(x, p=self.drop, training=self.training)
        x = F.relu(self.conv2(x, ei))
        x = F.dropout(x, p=self.drop, training=self.training)
        return self.lin(x)


def make_evaluate_gnn(data, gnn_seed=GNN_SEED):
    X_scaled          = data['X_scaled']
    y_all             = data['y']
    train_illicit_idx = data['train_illicit_idx']
    sel_n             = data['sel_n']
    ei_np             = data['ei_np']
    N_NODES           = data['N_NODES']
    test_mask         = data['test_mask']
    _LU               = np.full(N_NODES, -1, dtype=np.int32)

    def evaluate_gnn(top_n_illicit_local, label=''):
        """top_n_illicit_local: local indices into train_illicit_idx."""
        uniq            = np.unique(top_n_illicit_local)
        sel_anom_global = train_illicit_idx[uniq]
        test_global     = np.where(test_mask)[0]
        sub_nodes       = np.unique(np.concatenate([sel_n, sel_anom_global, test_global]))
        n_sub           = len(sub_nodes)

        _LU[:] = -1
        _LU[sub_nodes] = np.arange(n_sub)
        sl = _LU[ei_np[0]]; dl = _LU[ei_np[1]]
        ok = (sl >= 0) & (dl >= 0)
        sub_ei = torch.tensor([sl[ok], dl[ok]], dtype=torch.long).to(DEVICE)

        sub_X = torch.from_numpy(X_scaled[sub_nodes]).float().to(DEVICE)
        sub_y = torch.from_numpy(y_all[sub_nodes]).long().to(DEVICE)

        train_set = set(np.concatenate([sel_n, sel_anom_global]).tolist())
        test_set  = set(test_global.tolist())
        tr_gnn = torch.tensor([g in train_set for g in sub_nodes], dtype=torch.bool).to(DEVICE)
        te_gnn = torch.tensor([g in test_set  for g in sub_nodes], dtype=torch.bool).to(DEVICE)

        y_tr_sub = y_all[sub_nodes][tr_gnn.cpu().numpy()]
        n0 = int((y_tr_sub == 0).sum()); n1 = int((y_tr_sub == 1).sum())
        cw = torch.tensor([1.0, n0 / max(n1, 1)], dtype=torch.float).to(DEVICE)

        torch.manual_seed(gnn_seed)
        model = GCN(X_scaled.shape[1]).to(DEVICE)
        opt   = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

        best_loss, best_state, patience = float('inf'), None, 0
        for _ in range(GNN_EPOCHS):
            model.train(); opt.zero_grad()
            loss = F.cross_entropy(model(sub_X, sub_ei)[tr_gnn], sub_y[tr_gnn], weight=cw)
            loss.backward(); opt.step()
            if loss.item() < best_loss:
                best_loss = loss.item(); best_state = copy.deepcopy(model.state_dict()); patience = 0
            else:
                patience += 1
            if patience >= GNN_PATIENCE: break

        model.load_state_dict(best_state); model.eval()
        with torch.no_grad():
            prob = F.softmax(model(sub_X, sub_ei), dim=1)[:, 1].cpu().numpy()

        te_cpu  = te_gnn.cpu().numpy()
        y_te    = y_all[sub_nodes][te_cpu]
        prob_te = prob[te_cpu]
        res = {
            'PR-AUC': average_precision_score(y_te, prob_te),
            'F1':     f1_score(y_te, (prob_te >= 0.5).astype(int), zero_division=0),
            'AUC':    roc_auc_score(y_te, prob_te),
        }
        if label:
            print(f'  {label:<44} PR-AUC={res["PR-AUC"]:.4f}  F1={res["F1"]:.4f}  AUC={res["AUC"]:.4f}')
        return res

    return evaluate_gnn


print('GCN + evaluate_gnn factory 정의 완료')
print(f'  GNN_EPOCHS={GNN_EPOCHS}  GNN_PATIENCE={GNN_PATIENCE}')

GCN + evaluate_gnn factory 정의 완료
  GNN_EPOCHS=150  GNN_PATIENCE=15


---
## Section A — Exp 1b 재현: AISO as Graph Sampler

**연구 질문**: AISO가 도메인 피처 기반 node scoring으로 더 좋은 서브그래프를 찾는가?

**도메인 피처** (Dom-12 역할, 그래프 토폴로지에서 자동 계산):
- `degree`: 노드 연결 수
- `neighbor_fraud_ratio`: 이웃 중 fraud 비율 (train 레이블만)
- `neigh_mean_f0..f7`: 1-hop 이웃 raw feature 평균 (MI 상위 8개)

**메커니즘** (Exp 1b 동일):
- AISO W_i = 도메인 피처 클러스터 선호 분포
- Score = SGD(선택된 도메인 피처) → illicit 노드 스코어 → top-N 선택
- 다른 W_i → 다른 피처 관점 → 다른 종류의 illicit 노드 선택

**비교**: Random | Greedy(domain feats) | AISO(Rand M) | AISO(Smart M) — 모두 도메인 피처 기반

In [ ]:
def compute_domain_features(data, cfg):
    """
    Dom-12 equivalent: graph-topology features computed from adjacency.
    - degree
    - neighbor_fraud_ratio  (train labels only, no leakage)
    - top-8 1-hop neighbor raw feature means  (by MI with label)
    """
    N          = data['N_NODES']
    ei         = data['ei_np']       # (2, E)
    y          = data['y']
    train_mask = data['train_mask']
    X_scaled   = data['X_scaled']

    # Sparse adjacency from edge_index
    vals = np.ones(ei.shape[1], dtype=np.float32)
    adj  = csr_matrix((vals, (ei[0], ei[1])), shape=(N, N))

    # 1. Degree
    deg = np.array(adj.sum(axis=1), dtype=float).flatten()

    # 2. Neighbor fraud ratio  (use only train labels; test nodes treated as unlabeled)
    y_tr_only  = (y * train_mask).astype(float)
    tr_float   = train_mask.astype(float)
    n_fr       = np.array(adj @ y_tr_only).flatten()
    n_tr       = np.array(adj @ tr_float).flatten()
    neigh_fr   = n_fr / np.maximum(n_tr, 1)

    # 3. 1-hop mean of raw features  (adj_norm @ X_scaled)
    deg_safe = np.maximum(deg, 1)
    adj_norm = sp_diags(1.0 / deg_safe) @ adj
    neigh_mean = np.array(adj_norm @ X_scaled, dtype=float)  # (N, D)

    # Pick top features by MI with train labels
    mi       = mutual_info_classif(neigh_mean[train_mask], y[train_mask], random_state=42)
    top_idx  = np.argsort(mi)[::-1][:8]
    neigh_sel = neigh_mean[:, top_idx]   # (N, 8)

    # Stack → (N, 2+8)
    dom = np.column_stack([deg, neigh_fr, neigh_sel])
    dom = StandardScaler().fit_transform(dom)

    print(f'  도메인 피처: {dom.shape[1]}개 '
          f'(degree, neigh_fraud_ratio, top-{len(top_idx)} neigh_mean)')
    return dom


def random_node_selection(X_pool, n_illicit, seed):
    return np.random.RandomState(seed).choice(len(X_pool), n_illicit, replace=False)


def greedy_dom_selection(X_tr_dom, y_tr, X_pool_dom, n_illicit, seed):
    """Top-N by SGD on ALL domain features."""
    sgd = SGDClassifier(loss='log_loss', max_iter=100, tol=1e-4,
                         class_weight='balanced', random_state=seed)
    sgd.fit(X_tr_dom, y_tr)
    scores = sgd.predict_proba(X_pool_dom)[:, 1]
    return np.argsort(scores)[-n_illicit:]


print('Section A 유틸 정의 완료')
print('  compute_domain_features | random_node_selection | greedy_dom_selection')

In [ ]:
def run_exp1b(dataset_name, data, cfg):
    """
    Section A: AISO as graph sampler.
    도메인 피처(graph topology) 기반으로 node scoring → 서브그래프 선택.
    AISOwrapperGNN을 도메인 피처 공간에 적용 — Exp 7과 동일한 wrapper, 다른 피처 공간.
    """
    N_ILLICIT   = data['N_ILLICIT']
    K_DOM       = cfg['K_DOM']
    N_DOM_TYPES = cfg['N_DOM_TYPES']
    y_tr        = data['y_tr_all']
    train_mask  = data['train_mask']
    train_illicit_idx = data['train_illicit_idx']
    evaluate_gnn = make_evaluate_gnn(data)

    print(f'\n[{dataset_name}] Section A — Graph Sampler (도메인 피처 기반)')

    # ── 도메인 피처 계산 ─────────────────────────────────────────
    print(f'  도메인 피처 계산 중...')
    dom_all   = compute_domain_features(data, cfg)        # (N, D_dom)
    dom_pool  = dom_all[train_illicit_idx]                # illicit pool domain feats
    dom_tr    = dom_all[data['train_mask']]               # train domain feats

    # ── Smart M (도메인 피처 클러스터 기반) ──────────────────────
    M_smart_dom, _, dom_cl, dom_MI = build_feat_M(dom_tr, y_tr, N_DOM_TYPES)
    print(f'  도메인 M_smart: 클러스터={sorted(np.bincount(dom_cl).tolist())}')
    print(f'  비대칭 확인: M[0,1]={M_smart_dom[0,1]:.3f}, M[1,0]={M_smart_dom[1,0]:.3f}')

    results    = {m: [] for m in ['Random', 'Greedy (dom)',
                                   'AISO-dom (Rand M)', 'AISO-dom (Smart M)']}

    # ── Random ──────────────────────────────────────────────────
    for seed in SEEDS:
        top_n = random_node_selection(dom_pool, N_ILLICIT, seed)
        res   = evaluate_gnn(top_n, label=f'Random       seed={seed}')
        res['seed'] = seed; results['Random'].append(res)

    # ── Greedy (domain feats) ────────────────────────────────────
    for seed in SEEDS:
        top_n = greedy_dom_selection(dom_tr, y_tr, dom_pool, N_ILLICIT, seed)
        res   = evaluate_gnn(top_n, label=f'Greedy(dom)  seed={seed}')
        res['seed'] = seed; results['Greedy (dom)'].append(res)

    # ── AISO on domain features ──────────────────────────────────
    # 동일한 AISOwrapperGNN, 입력만 도메인 피처로 교체
    dom_wrapper = AISOwrapperGNN(
        n_agents=20, n_iter=60, beta=0.15, T_start=1.0, T_end=0.05,
        gnn_budget=GNN_BUDGET
    )

    for mname, M_cfg in [('AISO-dom (Rand M)', 'random'),
                          ('AISO-dom (Smart M)', M_smart_dom)]:
        print(f'\n  [{dataset_name}] {mname}')
        for seed in SEEDS:
            M_use = (np.random.RandomState(seed).uniform(-0.5, 2.0, (N_DOM_TYPES, N_DOM_TYPES))
                     if isinstance(M_cfg, str) else M_cfg)
            print(f'    seed={seed} ...', end='', flush=True)
            t0 = time.time()

            best_feat, top_n_best, timing, fid_cands = dom_wrapper.select(
                dom_tr, y_tr, dom_pool, dom_cl,
                K_DOM, N_ILLICIT, M_use,
                seed=seed, method_name=mname,
            )
            print(f' search={time.time()-t0:.0f}s  '
                  f'cache={timing["cache_size"]}  '
                  f'proxy={timing["best_proxy_auc"]:.4f}')

            top_n = fid_cands[0]['top_n_local'] if fid_cands else top_n_best
            res   = evaluate_gnn(top_n, label=f'{mname[:20]:20s} seed={seed}')
            res['seed'] = seed
            results[mname].append(res)

    return results


print('run_exp1b (도메인 피처 기반) 정의 완료')

---
## Section B — Exp 7 재현: AISO as Feature Engineer

**연구 질문**: AISO가 피처를 자동 선택해서 node scoring 품질을 높이는가?

- 노드 선택: **score-based top-N 고정** (선택된 피처 관점에서 SGD 스코어링)
- 변인: 피처 선택 전략
  - MI top-K: Mutual Information 상위 K개
  - mRMR top-K: max-Relevance min-Redundancy
  - AISO (Rand M): 랜덤 호환 행렬
  - AISO (Smart M): 피처 클러스터 기반 비대칭 M

In [6]:
def build_feat_M(X_tr, y_tr, N_TYPES, gamma=GAMMA):
    D = X_tr.shape[1]
    C_abs = np.abs(np.corrcoef(X_tr.T)); np.fill_diagonal(C_abs, 0.0)
    dist  = np.clip(1.0 - C_abs, 0.0, None); np.fill_diagonal(dist, 0.0)
    cl    = AgglomerativeClustering(n_clusters=N_TYPES,
                                     metric='precomputed',
                                     linkage='average').fit_predict(dist)
    print(f'  클러스터: {sorted(np.bincount(cl).tolist())}')
    MI    = mutual_info_classif(X_tr, y_tr, random_state=42)
    mean_MI = np.array([MI[cl==k].mean() if (cl==k).any() else 0.0 for k in range(N_TYPES)])
    MI_norm = (mean_MI - mean_MI.min()) / (mean_MI.max() - mean_MI.min() + 1e-8)

    def _M(mode):
        M = np.zeros((N_TYPES, N_TYPES))
        for i in range(N_TYPES):
            for j in range(N_TYPES):
                if i == j:
                    M[i][j] = -1.0
                else:
                    fi = np.where(cl==i)[0]; fj = np.where(cl==j)[0]
                    cp = -np.mean(C_abs[np.ix_(fi, fj)])
                    M[i][j] = cp + (gamma * (MI_norm[j] - MI_norm[i]) if mode == 'smart' else 0)
        return M

    return _M('smart'), _M('corr'), cl, MI


def mi_selection(X_tr, y_tr, X_pool, K, n_illicit, seed):
    mi   = mutual_info_classif(X_tr, y_tr, random_state=seed)
    feat = np.argsort(mi)[::-1][:K]
    sgd  = SGDClassifier(loss='log_loss', max_iter=100, tol=1e-4,
                          class_weight='balanced', random_state=seed)
    sgd.fit(X_tr[:, feat], y_tr)
    return feat, np.argsort(sgd.predict_proba(X_pool[:, feat])[:, 1])[-n_illicit:]


def mrmr_selection(X_tr, y_tr, X_pool, K, n_illicit, seed):
    mi    = mutual_info_classif(X_tr, y_tr, random_state=seed)
    Cmrmr = np.abs(np.corrcoef(X_tr.T)); np.fill_diagonal(Cmrmr, 0.0)
    sel   = [int(np.argmax(mi))]
    rem   = list(range(X_tr.shape[1])); rem.remove(sel[0])
    while len(sel) < K:
        sc   = [mi[f] - float(np.mean(Cmrmr[f, sel])) for f in rem]
        best = rem[int(np.argmax(sc))]
        sel.append(best); rem.remove(best)
    feat = np.array(sel)
    sgd  = SGDClassifier(loss='log_loss', max_iter=100, tol=1e-4,
                          class_weight='balanced', random_state=seed)
    sgd.fit(X_tr[:, feat], y_tr)
    return feat, np.argsort(sgd.predict_proba(X_pool[:, feat])[:, 1])[-n_illicit:]


def mask_jaccard(masks):
    if len(masks) < 2: return 0.0
    vals = []
    for i in range(len(masks)):
        for j in range(i+1, len(masks)):
            si, sj = set(masks[i]), set(masks[j])
            vals.append(len(si & sj) / max(len(si | sj), 1))
    return float(np.mean(vals))

def coverage_entropy(node_lists, n_total):
    if not node_lists: return 0.0
    freq = np.zeros(n_total)
    for nl in node_lists: np.add.at(freq, nl, 1)
    p = freq / max(freq.sum(), 1); p = p[p > 0]
    return float(-np.sum(p * np.log(p + 1e-10)))

def overlap_ratio(a, b):
    sa, sb = set(a), set(b)
    return len(sa & sb) / max(len(sa | sb), 1)


class AISOwrapperGNN:
    """P(S,F) = P(S|F)·P(F): W_i selects features, SGD scores nodes."""
    def __init__(self, n_agents=20, n_iter=60, beta=0.15,
                 subsample_ratio=0.15, val_ratio=0.2,
                 T_start=1.0, T_end=0.05, gnn_budget=GNN_BUDGET,
                 div_interval=DIV_INTERVAL):
        self.n_agents = n_agents; self.n_iter = n_iter; self.beta = beta
        self.subsample_ratio = subsample_ratio; self.val_ratio = val_ratio
        self.T_start = T_start; self.T_end = T_end
        self.gnn_budget = gnn_budget; self.div_interval = div_interval

    def select(self, X_tr, y_tr, X_pool, cluster_labels, K_select, n_illicit, M,
               seed=42, method_name=''):
        rng = np.random.RandomState(seed)
        N, D = X_tr.shape; K = M.shape[0]; nb = min(3, self.n_agents - 1)
        n_pool = len(X_pool)

        val_n = int(self.val_ratio * N)
        val_idx = rng.choice(N, val_n, replace=False)
        tr_mask = np.ones(N, bool); tr_mask[val_idx] = False
        X_val, y_val = X_tr[val_idx], y_tr[val_idx]
        X_t,   y_t   = X_tr[tr_mask], y_tr[tr_mask]
        sub_n = max(50, int(self.subsample_ratio * len(X_t)))

        cache = {}; n_calls = [0]

        def get_mask(Wi, t, explore=True):
            if explore:
                ratio  = t / max(1, self.n_iter - 1)
                T      = self.T_start * ((self.T_end / self.T_start) ** ratio)
                logits = Wi[cluster_labels] / T; logits -= logits.max()
                probs  = np.exp(logits); probs /= probs.sum()
                return tuple(rng.choice(D, K_select, replace=False, p=probs))
            else:
                return tuple(np.argsort(Wi[cluster_labels])[-K_select:])

        def get_score(mask_key, X_sub, y_sub):
            n_calls[0] += 1
            if mask_key not in cache:
                feat = list(mask_key)
                sgd  = SGDClassifier(loss='log_loss', max_iter=5, tol=None,
                                      class_weight='balanced', random_state=seed)
                try:
                    sgd.fit(X_sub[:, feat], y_sub)
                    proxy_auc     = roc_auc_score(y_val, sgd.predict_proba(X_val[:, feat])[:, 1])
                    illicit_p     = sgd.predict_proba(X_pool[:, feat])[:, 1]
                except Exception:
                    proxy_auc = 0.5; illicit_p = rng.rand(n_pool)
                cache[mask_key] = {'proxy_auc': proxy_auc, 'illicit_scores': illicit_p}
            return cache[mask_key]['proxy_auc']

        def top_n(mask_key):
            return np.argsort(cache[mask_key]['illicit_scores'])[-n_illicit:]

        W = rng.dirichlet(np.ones(K), self.n_agents)
        scores = np.zeros(self.n_agents)
        for i in range(self.n_agents):
            sub = rng.choice(len(X_t), sub_n, replace=False)
            mk  = get_mask(W[i], 0)
            scores[i] = get_score(mk, X_t[sub], y_t[sub])

        for t in range(self.n_iter):
            Cm = W @ M @ W.T; np.fill_diagonal(Cm, 0.0)
            sub = rng.choice(len(X_t), sub_n, replace=False)
            X_sub, y_sub = X_t[sub], y_t[sub]
            for i in range(self.n_agents):
                att   = np.argsort(Cm[i])[-nb:]
                att_s = np.array([get_score(get_mask(W[j], t), X_sub, y_sub) for j in att])
                max_s = att_s.max() + 1e-8
                bj    = att[np.argmax(Cm[i, att] * att_s / max_s)]
                W_new = (1 - self.beta) * W[i] + self.beta * W[bj]
                W[i]  = W_new / W_new.sum()
                scores[i] = get_score(get_mask(W[i], t), X_sub, y_sub)

        # Final LR re-score
        final_lr = LogisticRegression(solver='liblinear', C=1.0, max_iter=500,
                                       class_weight='balanced', random_state=seed)
        fseen, final_scores = {}, []
        for i in range(self.n_agents):
            mk = get_mask(W[i], self.n_iter, explore=False); feat = list(mk)
            if mk not in fseen:
                try:
                    final_lr.fit(X_t[:, feat], y_t)
                    fseen[mk] = roc_auc_score(y_val, final_lr.predict_proba(X_val[:, feat])[:, 1])
                except Exception:
                    fseen[mk] = 0.5
                if mk not in cache:
                    get_score(mk, X_t, y_t)
            final_scores.append(fseen[mk])

        seen_keys, fidelity_cands = set(), []
        for ai in np.argsort(final_scores)[::-1]:
            mk = get_mask(W[ai], self.n_iter, explore=False)
            if mk not in seen_keys and mk in cache:
                seen_keys.add(mk)
                fidelity_cands.append({'feat_mask': np.array(sorted(mk)),
                                        'top_n_local': top_n(mk),
                                        'proxy_auc': cache[mk]['proxy_auc']})
            if len(fidelity_cands) >= self.gnn_budget: break

        best_mk   = get_mask(W[np.argmax(final_scores)], self.n_iter, explore=False)
        best_feat = np.array(sorted(best_mk))
        sgd_fin   = SGDClassifier(loss='log_loss', max_iter=100, tol=1e-4,
                                   class_weight='balanced', random_state=seed)
        try:
            sgd_fin.fit(X_t[:, list(best_mk)], y_t)
            top_n_best = np.argsort(sgd_fin.predict_proba(X_pool[:, list(best_mk)])[:, 1])[-n_illicit:]
        except Exception:
            top_n_best = top_n(best_mk)

        timing = {
            'cache_size': len(cache),
            'n_proxy_calls': n_calls[0],
            'best_proxy_auc': final_scores[np.argmax(final_scores)],
        }
        return best_feat, top_n_best, timing, fidelity_cands


wrapper = AISOwrapperGNN(n_agents=20, n_iter=60, beta=0.15, T_start=1.0, T_end=0.05)
print('Section B 정의 완료')

Section B 정의 완료


In [7]:
def run_exp7(dataset_name, data, cfg):
    """Section B: AISO as feature engineer. Score-based top-N node selection."""
    X_pool    = data['X_illicit_pool']
    X_tr      = data['X_tr_all']
    y_tr      = data['y_tr_all']
    K_SELECT  = cfg['K_SELECT']
    N_TYPES   = cfg['N_TYPES']
    N_ILLICIT = data['N_ILLICIT']
    D         = data['D']
    evaluate_gnn = make_evaluate_gnn(data)

    METHOD_ORDER = ['MI top-K', 'mRMR top-K', 'AISO (Rand M)', 'AISO (Smart M)']
    results      = {m: [] for m in METHOD_ORDER}
    feat_masks   = {m: [] for m in METHOD_ORDER}

    # Build Smart M
    print(f'\n[{dataset_name}] Section B — Feature Engineer')
    print(f'  Smart M 구성 중 (N_TYPES={N_TYPES}) ...')
    M_smart, _, cluster_labels, MI_global = build_feat_M(X_tr, y_tr, N_TYPES)
    print(f'  M_smart 비대칭 확인: M[0,1]={M_smart[0,1]:.3f}, M[1,0]={M_smart[1,0]:.3f}')

    # Baselines
    print(f'\n  [Baselines]')
    for seed in SEEDS:
        feat, top_n = mi_selection(X_tr, y_tr, X_pool, K_SELECT, N_ILLICIT, seed)
        feat_masks['MI top-K'].append(feat)
        res = evaluate_gnn(top_n, label=f'MI top-K       seed={seed}')
        res['seed'] = seed; results['MI top-K'].append(res)

    for seed in SEEDS:
        feat, top_n = mrmr_selection(X_tr, y_tr, X_pool, K_SELECT, N_ILLICIT, seed)
        feat_masks['mRMR top-K'].append(feat)
        res = evaluate_gnn(top_n, label=f'mRMR top-K     seed={seed}')
        res['seed'] = seed; results['mRMR top-K'].append(res)

    # AISO
    for mname, M_cfg in [('AISO (Rand M)', 'random'), ('AISO (Smart M)', M_smart)]:
        print(f'\n  [{dataset_name}] {mname}')
        for seed in SEEDS:
            M_use = (np.random.RandomState(seed).uniform(-0.5, 2.0, (N_TYPES, N_TYPES))
                     if isinstance(M_cfg, str) else M_cfg)
            print(f'    seed={seed} ...', end='', flush=True)
            t0 = time.time()
            best_feat, top_n_best, timing, fid_cands = wrapper.select(
                X_tr, y_tr, X_pool, cluster_labels, K_SELECT, N_ILLICIT,
                M_use, seed=seed, method_name=mname
            )
            feat_masks[mname].append(best_feat)
            print(f' search={time.time()-t0:.0f}s  '
                  f'cache={timing["cache_size"]}  '
                  f'proxy={timing["best_proxy_auc"]:.4f}')

            # GNN on best candidate (fidelity_cands[0] = highest proxy)
            if fid_cands:
                res = evaluate_gnn(fid_cands[0]['top_n_local'],
                                   label=f'{mname[:20]:20s} seed={seed}')
            else:
                res = evaluate_gnn(top_n_best,
                                   label=f'{mname[:20]:20s} seed={seed}')
            res['seed'] = seed
            results[mname].append(res)

    return results, feat_masks, cluster_labels, M_smart


print('run_exp7 정의 완료')

run_exp7 정의 완료


---
## 3. 실험 실행

In [8]:
print('=' * 65)
print('YelpChi  (32 features, ~45k nodes)')
print('=' * 65)

yelp_1b = run_exp1b('YelpChi', yelp_data, DATASET_CONFIGS['YelpChi'])
yelp_7, yelp_feat_masks, yelp_cl, yelp_M = run_exp7('YelpChi', yelp_data, DATASET_CONFIGS['YelpChi'])

YelpChi  (32 features, ~45k nodes)

[YelpChi] Section A — Graph Sampler


C:\Users\kevin\AppData\Local\Temp\ipykernel_30868\763960594.py:39: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  sub_ei = torch.tensor([sl[ok], dl[ok]], dtype=torch.long).to(DEVICE)


  Random          seed=0                       PR-AUC=0.2546  F1=0.2851  AUC=0.6125
  Random          seed=7                       PR-AUC=0.2350  F1=0.2749  AUC=0.6015
  Random          seed=42                      PR-AUC=0.2403  F1=0.2750  AUC=0.6041
  Random          seed=77                      PR-AUC=0.2248  F1=0.2741  AUC=0.5981
  Random          seed=123                     PR-AUC=0.2395  F1=0.2753  AUC=0.6093
  Greedy          seed=0                       PR-AUC=0.2245  F1=0.2610  AUC=0.5837
  Greedy          seed=7                       PR-AUC=0.2195  F1=0.2638  AUC=0.5822
  Greedy          seed=42                      PR-AUC=0.2266  F1=0.2507  AUC=0.5813
  Greedy          seed=77                      PR-AUC=0.2249  F1=0.2566  AUC=0.5754
  Greedy          seed=123                     PR-AUC=0.2284  F1=0.2627  AUC=0.5803
  Cluster-uniform seed=0                       PR-AUC=0.2299  F1=0.2762  AUC=0.5986
  Cluster-uniform seed=7                       PR-AUC=0.2322  F1=0.2858  AUC

In [9]:
print('=' * 65)
print('Amazon  (25 features, ~12k nodes)')
print('=' * 65)

amzn_1b = run_exp1b('Amazon', amzn_data, DATASET_CONFIGS['Amazon'])
amzn_7, amzn_feat_masks, amzn_cl, amzn_M = run_exp7('Amazon', amzn_data, DATASET_CONFIGS['Amazon'])

Amazon  (25 features, ~12k nodes)

[Amazon] Section A — Graph Sampler
  Random          seed=0                       PR-AUC=0.5134  F1=0.3528  AUC=0.8933
  Random          seed=7                       PR-AUC=0.4205  F1=0.3575  AUC=0.8744
  Random          seed=42                      PR-AUC=0.4903  F1=0.3750  AUC=0.8842
  Random          seed=77                      PR-AUC=0.5201  F1=0.3492  AUC=0.8923
  Random          seed=123                     PR-AUC=0.4269  F1=0.3354  AUC=0.8854
  Greedy          seed=0                       PR-AUC=0.5146  F1=0.4354  AUC=0.8756
  Greedy          seed=7                       PR-AUC=0.4985  F1=0.4170  AUC=0.8791
  Greedy          seed=42                      PR-AUC=0.5203  F1=0.4272  AUC=0.8692
  Greedy          seed=77                      PR-AUC=0.5141  F1=0.4426  AUC=0.8712
  Greedy          seed=123                     PR-AUC=0.5254  F1=0.4481  AUC=0.8730
  Cluster-uniform seed=0                       PR-AUC=0.4477  F1=0.3104  AUC=0.8721
  Clus

---
## 4. 결과 요약

In [ ]:
def summarize(results):
    s = {}
    for m, rlist in results.items():
        prs  = [r['PR-AUC'] for r in rlist]
        f1s  = [r['F1']     for r in rlist]
        aucs = [r['AUC']    for r in rlist]
        s[m] = dict(mean=np.mean(prs), std=np.std(prs),
                    f1=np.mean(f1s),   auc=np.nanmean(aucs))
    return s


def print_table(title, summ_1b, summ_7):
    print(f'\n=== {title} ===')
    ORDER_A = ['Random', 'Greedy (dom)', 'AISO-dom (Rand M)', 'AISO-dom (Smart M)']
    ORDER_B = ['MI top-K', 'mRMR top-K', 'AISO (Rand M)', 'AISO (Smart M)']

    print(f'\n  [Section A — Graph Sampler]  고정 피처: ALL')
    print(f'  {"Method":<26} {"PR-AUC":>8} {"±std":>7}  {"F1":>7}')
    print('  ' + '-' * 54)
    for m in ORDER_A:
        if m not in summ_1b: continue
        s = summ_1b[m]
        print(f'  {m:<26} {s["mean"]:>8.4f} {s["std"]:>7.4f}  {s["f1"]:>7.4f}')

    print(f'\n  [Section B — Feature Engineer]  노드: score-based top-N')
    print(f'  {"Method":<26} {"PR-AUC":>8} {"±std":>7}  {"F1":>7}')
    print('  ' + '-' * 54)
    for m in ORDER_B:
        if m not in summ_7: continue
        s = summ_7[m]
        print(f'  {m:<26} {s["mean"]:>8.4f} {s["std"]:>7.4f}  {s["f1"]:>7.4f}')


yelp_s1b = summarize(yelp_1b)
yelp_s7  = summarize(yelp_7)
amzn_s1b = summarize(amzn_1b)
amzn_s7  = summarize(amzn_7)

print_table('YelpChi', yelp_s1b, yelp_s7)
print_table('Amazon',  amzn_s1b, amzn_s7)

In [ ]:
# Cross-dataset summary: AISO Smart M vs best baseline per section
print('\n=== 3-Dataset Cross Comparison ===')
print('\n  [Section A: Graph Sampler]  AISO-dom (Smart M) vs Greedy (dom)')
print(f'  {"Dataset":<10} {"Greedy(dom)":>11} {"AISO-dom(SM)":>13} {"Δ":>6}')
print('  ' + '-' * 46)
for dname, s1b in [("YelpChi", yelp_s1b), ("Amazon", amzn_s1b)]:
    g  = s1b.get('Greedy (dom)',          {}).get('mean', float('nan'))
    sm = s1b.get('AISO-dom (Smart M)',    {}).get('mean', float('nan'))
    print(f'  {dname:<10} {g:>11.4f} {sm:>13.4f} {sm-g:>+6.4f}')

print('\n  [Section B: Feature Engineer]  AISO (Smart M) vs mRMR')
print(f'  {"Dataset":<10} {"mRMR top-K":>10} {"AISO(SM)":>10} {"Δ":>6}')
print('  ' + '-' * 40)
for dname, s7 in [("YelpChi", yelp_s7), ("Amazon", amzn_s7)]:
    mr = s7.get('mRMR top-K',    {}).get('mean', float('nan'))
    sm = s7.get('AISO (Smart M)', {}).get('mean', float('nan'))
    print(f'  {dname:<10} {mr:>10.4f} {sm:>10.4f} {sm-mr:>+6.4f}')

# Load Elliptic results for 3-dataset comparison
try:
    _e = pd.read_csv(BASE / 'exp7_gnn_evals.csv')
    def _elliptic_mean(method):
        sub = _e[_e['method'] == method]
        if sub.empty: return float('nan'), float('nan')
        vals = []
        for seed in [0, 7, 42, 77, 123]:
            ss = sub[sub['seed'] == seed]
            if ss.empty: continue
            if ss['proxy_auc'].notna().any():
                best = ss.loc[ss['proxy_auc'].idxmax(), 'gnn_pr_auc']
            else:
                best = ss['gnn_pr_auc'].max()
            vals.append(best)
        return np.mean(vals), np.std(vals)

    e_smart_mean, e_smart_std = _elliptic_mean('AISO (Smart M)')
    e_mrmr_mean,  _            = _elliptic_mean('mRMR top-K')
    print(f'  {"Elliptic":<10} {e_mrmr_mean:>10.4f} {e_smart_mean:>10.4f} {e_smart_mean-e_mrmr_mean:>+6.4f}  (from exp7_gnn_evals.csv)')
except FileNotFoundError:
    print('  (exp7_gnn_evals.csv not found — run Exp 7 Elliptic notebook first)')

In [ ]:
# Visualization
colors_A = {'Random': '#bdc3c7', 'Greedy (dom)': '#3498db',
            'AISO-dom (Rand M)': '#e74c3c', 'AISO-dom (Smart M)': '#2ecc71'}
colors_B = {'MI top-K': '#3498db', 'mRMR top-K': '#f39c12',
            'AISO (Rand M)': '#e74c3c', 'AISO (Smart M)': '#2ecc71'}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

def plot_bar(ax, summ, order, colors, title):
    ms = [m for m in order if m in summ]
    mn = [summ[m]['mean'] for m in ms]
    st = [summ[m]['std']  for m in ms]
    cl = [colors.get(m, '#95a5a6') for m in ms]
    x  = np.arange(len(ms))
    ax.bar(x, mn, yerr=st, capsize=5, color=cl, alpha=0.85, width=0.65)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(' ', '\n') for m in ms], fontsize=8)
    ax.set_ylabel('PR-AUC'); ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.3, axis='y')
    for xi, (mv, sv) in enumerate(zip(mn, st)):
        ax.text(xi, mv + sv + 0.003, f'{mv:.4f}', ha='center', va='bottom', fontsize=7)

ORDER_A = ['Random', 'Greedy (dom)', 'AISO-dom (Rand M)', 'AISO-dom (Smart M)']
ORDER_B = ['MI top-K', 'mRMR top-K', 'AISO (Rand M)', 'AISO (Smart M)']

plot_bar(axes[0,0], yelp_s1b, ORDER_A, colors_A, 'YelpChi — Section A (Graph Sampler, dom feats)')
plot_bar(axes[0,1], yelp_s7,  ORDER_B, colors_B, 'YelpChi — Section B (Feature Engineer)')
plot_bar(axes[1,0], amzn_s1b, ORDER_A, colors_A, 'Amazon  — Section A (Graph Sampler, dom feats)')
plot_bar(axes[1,1], amzn_s7,  ORDER_B, colors_B, 'Amazon  — Section B (Feature Engineer)')

plt.suptitle(
    'AISO: Graph Sampler (1b) vs Feature Engineer (7)\n'
    'YelpChi & Amazon Multi-Dataset Validation',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(BASE / 'exp_fraud_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('→ exp_fraud_results.png 저장')

In [13]:
# Save CSVs
rows = []
for dname, r1b, r7 in [('YelpChi', yelp_1b, yelp_7), ('Amazon', amzn_1b, amzn_7)]:
    for section, rdict in [('A_node_sampler', r1b), ('B_feat_engineer', r7)]:
        for method, rlist in rdict.items():
            for r in rlist:
                rows.append({
                    'dataset': dname, 'section': section, 'method': method,
                    'seed': r.get('seed', -1),
                    'gnn_pr_auc': r['PR-AUC'], 'gnn_f1': r['F1'], 'gnn_auc': r['AUC'],
                })

df_out = pd.DataFrame(rows)
df_out.to_csv(BASE / 'exp_fraud_gnn_evals.csv', index=False)
print(f'exp_fraud_gnn_evals.csv: {len(df_out)} rows')
print(df_out.groupby(['dataset','section','method'])['gnn_pr_auc'].agg(['mean','std']).round(4))

exp_fraud_gnn_evals.csv: 90 rows
                                               mean     std
dataset section         method                             
Amazon  A_node_sampler  AISO-node (Rand M)   0.3093  0.0299
                        AISO-node (Smart M)  0.3512  0.0182
                        Cluster-uniform      0.4440  0.0230
                        Greedy               0.5146  0.0101
                        Random               0.4743  0.0475
        B_feat_engineer AISO (Rand M)        0.4035  0.0765
                        AISO (Smart M)       0.5067  0.0364
                        MI top-K             0.5051  0.0063
                        mRMR top-K           0.5140  0.0055
YelpChi A_node_sampler  AISO-node (Rand M)   0.1926  0.0065
                        AISO-node (Smart M)  0.1890  0.0027
                        Cluster-uniform      0.2375  0.0100
                        Greedy               0.2248  0.0033
                        Random               0.2388  0.0108
       